# Module 5: Difference in Differences, by Hand

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Four numbers and one division. This module computes a difference in
differences from raw counts, on paper, so that nothing later in the series is
a black box.

It also settles a question most treatments skip: **whether to subtract or to
divide.** The two give different answers, and on this dataset only one of
them is right.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

## 2. The four cells, in raw counts

Start with incidents and arrests rather than rates, so that nothing is hidden.

In [ ]:
keep = [a for a in TRAINED if a != "A007"]          # Summit County set aside, Module 7

rows = []
for label, ids in [("took the training", keep), ("did not", COMPARISON)]:
    for per in ["before", "after"]:
        d = f[f["agency_id"].isin(ids) & (f["period"] == per)]
        rows.append({"group": label, "period": per,
                     "use of force": int(d["n_uof"].sum()),
                     "arrests": int(d["n_arrests"].sum()),
                     "rate per 100": round(100 * d["n_uof"].sum() / d["n_arrests"].sum(), 3)})
pd.DataFrame(rows).set_index(["group", "period"])

Every later number in this module comes out of those four rates. Nothing else
is used.

## 3. The first difference, twice

There are two ways to say how much a group changed, and they are not
interchangeable.

In [ ]:
tb, ta = cell_rate(keep, "before"), cell_rate(keep, "after")
cb, ca = cell_rate(COMPARISON, "before"), cell_rate(COMPARISON, "after")

print("  trained agencies")
print(f"    subtracted: {ta:.3f} - {tb:.3f} = {ta - tb:+.3f} rate points")
print(f"    divided:    {ta:.3f} / {tb:.3f} = {ta / tb:.4f}, "
      f"which is {100 * (ta / tb - 1):+.1f} percent")
print("  comparison agencies")
print(f"    subtracted: {ca:.3f} - {cb:.3f} = {ca - cb:+.3f} rate points")
print(f"    divided:    {ca:.3f} / {cb:.3f} = {ca / cb:.4f}, "
      f"which is {100 * (ca / cb - 1):+.1f} percent")

The comparison group fell by **0.52 rate points** and by **19.5 percent**.
Both statements describe the same two numbers, and they imply different
counterfactuals for a group that started at a different level.

## 4. The second difference, twice

In [ ]:
add_y0 = tb + (ca - cb)          # same change in rate points
mul_y0 = tb * (ca / cb)          # same proportional change

print(f"  the trained agencies started at {tb:.3f} and ended at {ta:.3f}\n")
print(f"  Y(0) if they had fallen by the same 0.52 rate points:  {add_y0:.3f}")
print(f"    estimate: {ta:.3f} - {add_y0:.3f} = {ta - add_y0:+.3f} points, "
      f"which is {100 * (ta - add_y0) / tb:+.1f} percent of where they started")
print(f"\n  Y(0) if they had fallen by the same 19.5 percent:      {mul_y0:.3f}")
print(f"    estimate: {ta:.3f} / {mul_y0:.3f} = {ta / mul_y0:.4f}, "
      f"which is {100 * (ta / mul_y0 - 1):+.1f} percent")
print(f"\n  the truth: {TRUTH:+.1f} percent")

**Subtracting gives 15.0 percent. Dividing gives 12.5 percent.** The truth is
12.0.

Neither is a mistake in arithmetic. They answer to different assumptions about
what "the same change" means, and the difference matters because the two
groups started at different levels: 3.59 against 2.67.

The dataset's program was built to act on the **rate**, multiplying it by
0.88. So the proportional version recovers it and the additive version does
not. In real work you do not know which, and the choice has to be argued.

| Use the proportional version when | Use the additive version when |
|---|---|
| the two groups start at different levels | the groups are already similar |
| the intervention plausibly scales with volume | the intervention has a fixed size |
| the outcome is a rate or a count | the outcome is already a difference |

**Say which one you used.** A report that gives 15.0 and one that gives 12.5
are both defensible and they are not the same claim.

## 5. The whole thing on one line

In [ ]:
did = 100 * ((ta / tb) / (ca / cb) - 1)
print(f"  ({ta:.3f}/{tb:.3f}) / ({ca:.3f}/{cb:.3f}) - 1 = {did / 100:+.4f}"
      f"  ->  {did:+.1f} percent")

That is the entire method. Everything in [Module 6](Module_06_Difference_In_Differences_As_A_Regression.ipynb)
is a way of getting the same number with an interval attached, and everything
after that is a way of checking whether it means anything.

## Exercise

The phase in months, July to October 2023, were left out of both periods.
Find out what including them would have done.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for label, after_periods, before_periods in [
            ("phase in months excluded", ["after"], ["before"]),
            ("phase in counted as after", ["after", "phase"], ["before"]),
            ("phase in counted as before", ["after"], ["before", "phase"])]:
        tb2 = rate(f[f["agency_id"].isin(keep) & f["period"].isin(before_periods)])
        ta2 = rate(f[f["agency_id"].isin(keep) & f["period"].isin(after_periods)])
        cb2 = rate(f[f["agency_id"].isin(COMPARISON) & f["period"].isin(before_periods)])
        ca2 = rate(f[f["agency_id"].isin(COMPARISON) & f["period"].isin(after_periods)])
        rows.append({"treatment of the phase in": label,
                     "estimate": f"{100 * ((ta2 / tb2) / (ca2 / cb2) - 1):+.2f}%"})
    rows.append({"treatment of the phase in": "THE TRUTH",
                 "estimate": f"{TRUTH:+.2f}%"})
    display(pd.DataFrame(rows).set_index("treatment of the phase in"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

All three land within a few tenths of a point of each other. The phase in is
four months against a 30 month after period and a 54 month before period, so
whichever side it is assigned to, it is diluted.

**That is a fact about this dataset, not a general rule.** With a two year
phase in and a one year follow up, the three answers would differ
substantially, and the one that counts a partially treated period as fully
treated would understate the effect.

The habit worth keeping is the one this module used by default: **give the
transition its own period** rather than forcing it into one side. It costs one
indicator, it makes the assumption visible, and it removes a choice that would
otherwise be made silently.

</details>

---

**Next:** [Module 6: Difference in Differences, as a Regression](Module_06_Difference_In_Differences_As_A_Regression.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*